# Lesson 9: Feature Engineering & Preprocessing

**Feature engineering** = preparing and improving the input data (the features) so a model can learn from it well.

Real-world data is messy: missing values, text categories, wildly different scales. Models can't handle that raw. This lesson is the cleanup + improvement step — often the biggest lever on model quality.

We'll cover, one keyword at a time:
1. **Missing values** (and how to fill them = 'imputation')
2. **Encoding** categories (turning text into numbers)
3. **Feature scaling** (putting features on a comparable range)
4. **Creating new features**

> Upload to [Google Colab](https://colab.research.google.com) and run top to bottom.

## Step 0: A small messy dataset to work with

In [ ]:
import pandas as pd
import numpy as np

# A tiny dataset about houses. Notice: missing values (NaN) and a TEXT column.
df = pd.DataFrame({
    'size_sqft':   [1500, 2000, np.nan, 1800, 1200, 2500],
    'bedrooms':    [3, 4, 2, 3, np.nan, 4],
    'city':        ['Austin', 'Dallas', 'Austin', 'Houston', 'Dallas', 'Houston'],
    'price':       [300000, 450000, 210000, 360000, 175000, 540000],
})
print(df)
print('\nMissing values per column:')
print(df.isnull().sum())

## Step 1: Missing values -> Imputation

**Imputation** = filling in missing values with a sensible substitute (instead of deleting the row).

Common strategies: fill numeric columns with the **mean** or **median**; fill category columns with the **most frequent** value.

In [ ]:
from sklearn.impute import SimpleImputer

# Fill missing numeric values with the column's MEAN
num_cols = ['size_sqft', 'bedrooms']
imputer = SimpleImputer(strategy='mean')
df[num_cols] = imputer.fit_transform(df[num_cols])

print('After imputation (no more NaN):')
print(df)

## Step 2: Encoding categories -> turning text into numbers

Models only understand numbers, not text like 'Austin'. **Encoding** converts categories into numbers.

**One-Hot Encoding** = make a new 0/1 column for each category. (Best for categories with no natural order, like cities.)

In [ ]:
# One-hot encode the 'city' text column
df_encoded = pd.get_dummies(df, columns=['city'], prefix='city')
print('Each city becomes its own 0/1 column:')
print(df_encoded)

## Step 3: Feature Scaling -> a fair range for every feature

`size_sqft` (~1000s) and `bedrooms` (~single digits) live on very different scales. Distance- and weight-based models (linear, logistic, KNN, PCA, K-Means) can be dominated by the big-numbered feature. **Scaling** fixes this.

- **Standardization (StandardScaler)**: rescales each feature to mean 0, standard deviation 1.
- **Normalization (MinMaxScaler)**: rescales each feature to a 0-1 range.

(Reminder: tree models do NOT need this; linear models, PCA and K-Means do.)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaled = scaler.fit_transform(df_encoded[['size_sqft', 'bedrooms']])

scaled_df = pd.DataFrame(scaled, columns=['size_scaled', 'bedrooms_scaled'])
print('After standardization (mean ~0, std ~1):')
print(scaled_df.round(2))
print('\nMeans:', scaled.mean(axis=0).round(2), ' Stds:', scaled.std(axis=0).round(2))

## Step 4: Creating new features (the creative part)

Sometimes a NEW feature built from existing ones carries more signal. Example: `price_per_sqft` may predict better than price and size separately.

In [ ]:
df_encoded['price_per_sqft'] = df_encoded['price'] / df_encoded['size_sqft']
print(df_encoded[['size_sqft', 'price', 'price_per_sqft']].round(1))

## Key takeaways

- **Imputation**: fill missing values (mean/median/most-frequent) instead of dropping data.
- **Encoding**: turn text categories into numbers (one-hot = a 0/1 column per category).
- **Scaling**: put features on a comparable range (needed for linear/distance models, not trees).
- **Feature creation**: combine existing features into more informative ones.
- This step is where the 'data quality + features' differentiators (your earlier insight) actually happen.

## Your turn

1. Change the imputation strategy to 'median'. Do the filled values change?
2. Try `MinMaxScaler` instead of `StandardScaler`. What range do the values fall into now?
3. Why would one-hot encoding be better than just numbering cities 1, 2, 3?
4. Think of one new feature you could create for predicting house price.